# Visualizacao com t-SNE

Neste notebook vamos carregar as matrizes **Bag of Words** e **TF-IDF** geradas no notebook `introducao_pln` e aplicar o **t-SNE** para reduzir a dimensionalidade a duas dimensoes. Assim conseguimos visualizar como as noticias se agrupam segundo cada representacao.

## O que e t-SNE

O **t-SNE** (*t-distributed Stochastic Neighbor Embedding*) e uma tecnica de reducao de dimensionalidade nao linear, muito usada para visualizacao. Ele tenta colocar pontos parecidos no espacao original proximos no espaco reduzido (em geral 2D), preservando a estrutura local dos dados.

Aqui, cada ponto sera uma noticia. Esperamos que noticias com vocabulario parecido apareçam proximas no grafico.

In [1]:
import pandas as pd
import plotly.express as px
from sklearn.manifold import TSNE

## Carregando os dados

In [2]:
df_bow = pd.read_csv("../dados/bow.csv")
df_tfidf = pd.read_csv("../dados/tfidf.csv")

print(f"BoW:    {df_bow.shape}")
print(f"TF-IDF: {df_tfidf.shape}")
df_bow.head()

BoW:    (271, 10904)
TF-IDF: (271, 10903)


,data,tags,titulo,url,bow_aab,bow_aapen,bow_aasap,bow_abaixo,bow_abalos,bow_abandona,...,bow_zenaide,bow_zendron,bow_zequinha,bow_zerbone,bow_zero,bow_zettel,bow_zimbardo,bow_zona,bow_zumbi,palavras_unicas
0,2026-03-30T17:31:28-03:00,"['Agricultura familiar ', 'Emendas parlamentar...",Chico Rodrigues destaca ações em Roraima e cri...,https://www12.senado.leg.br/noticias/materias/...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,160
1,2026-03-27T19:39:12-03:00,"['Aposentados', 'INSS', 'Pensionistas']","Com mais de 4 mil páginas, relatório da CPMI p...",https://www12.senado.leg.br/noticias/materias/...,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,590
2,2026-03-24T16:59:17-03:00,"['Alexandre de Moraes', 'Jair Bolsonaro', 'Pro...",Wilder defende prisão domiciliar para Jair Bol...,https://www12.senado.leg.br/noticias/materias/...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,126
3,2025-12-16T18:54:41-03:00,"['Alexandre de Moraes', 'Supremo Tribunal Fede...",Girão aponta 'exposição' de ministros do STF e...,https://www12.senado.leg.br/noticias/materias/...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,159
4,2025-12-15T17:37:19-03:00,"['Ditadura militar', 'Jair Bolsonaro', 'Suprem...",Mecias defende anistia e pede prisão domicilia...,https://www12.senado.leg.br/noticias/materias/...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,148


## Separando metadados e matrizes

As colunas que comecam com `bow_` ou `tfidf_` sao as variaveis numericas. As outras sao metadados (titulo, url, tags, data).

In [3]:
colunas_bow = [c for c in df_bow.columns if c.startswith("bow_")]
colunas_tfidf = [c for c in df_tfidf.columns if c.startswith("tfidf_")]

X_bow = df_bow[colunas_bow].values
X_tfidf = df_tfidf[colunas_tfidf].values

titulos = df_bow["titulo"].tolist()

print(f"X_bow:   {X_bow.shape}")
print(f"X_tfidf: {X_tfidf.shape}")

X_bow:   (271, 10899)
X_tfidf: (271, 10899)


## Aplicando o t-SNE

Como temos poucas noticias (16), usamos `perplexity` baixo (o valor precisa ser menor que o numero de amostras).

In [4]:
perplexity = 5
random_state = 42

tsne_bow = TSNE(n_components=2, perplexity=perplexity, random_state=random_state, init="random")
embed_bow = tsne_bow.fit_transform(X_bow)

tsne_tfidf = TSNE(n_components=2, perplexity=perplexity, random_state=random_state, init="random")
embed_tfidf = tsne_tfidf.fit_transform(X_tfidf)

print("embed_bow:  ", embed_bow.shape)
print("embed_tfidf:", embed_tfidf.shape)

embed_bow:   (271, 2)
embed_tfidf: (271, 2)


## Visualizando lado a lado

In [5]:
def plotar(embed, nome):
    df_plot = pd.DataFrame({
        "x": embed[:, 0],
        "y": embed[:, 1],
        "indice": df_bow.index,
        "titulo": df_bow["titulo"],
        "data": df_bow["data"],
    })
    fig = px.scatter(
        df_plot,
        x="x",
        y="y",
        hover_data={"indice": True, "titulo": True, "data": True, "x": False, "y": False},
        title=f"t-SNE - {nome}",
        width=700,
        height=700,
    )
    fig.show()


plotar(embed_bow, "Bag of Words")
plotar(embed_tfidf, "TF-IDF")